<a href="https://colab.research.google.com/github/eminahamamdzic/FlyRank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminahamamdzic/FlyRank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, train_test_split

# 1. Colab setup i Git kloniranje
in_colab = "google.colab" in str(get_ipython())
if in_colab and not Path("FlyRank").exists():
    !git clone https://github.com/eminahamamdzic/FlyRank.git
    %cd FlyRank

# 2. Lociranje podataka u data/raw
candidate_paths = [
    Path("data/raw"),
    Path("../data/raw"),
    Path("../../data/raw"),
    Path("data"),
]
data_file = None
for cp in candidate_paths:
    if cp.exists():
        files = [
            f
            for f in list(cp.glob("*.csv")) + list(cp.glob("*.parquet"))
            if "baseline" not in f.name
        ]
        if files:
            data_file = files[0]
            break

print(f"✓ Učitavanje skupa podataka iz: {data_file}")
df = (
    pd.read_parquet(data_file)
    if data_file.suffix == ".parquet"
    else pd.read_csv(data_file)
)


# Standardizacija kolona
def find_col(candidates, dataframe):
    for c in candidates:
        for col in dataframe.columns:
            if col.lower() == c.lower():
                return col
    return None


col_vol = find_col(["impressions", "search_volume", "volume"], df)
col_pos = find_col(["position", "avg_position", "rank"], df)
col_clicks = find_col(["clicks", "click_count"], df)
col_ctr = find_col(["ctr", "click_through_rate"], df)
col_group = find_col(["client_id", "domain", "category", "content_type"], df)

df["impressions"] = (
    pd.to_numeric(df[col_vol], errors="coerce").fillna(0) if col_vol else 100.0
)
df["position"] = (
    pd.to_numeric(df[col_pos], errors="coerce").fillna(10.0)
    if col_pos
    else 10.0
)
df["clicks"] = (
    pd.to_numeric(df[col_clicks], errors="coerce").fillna(0)
    if col_clicks
    else 0.0
)

if col_ctr:
    df["ctr"] = pd.to_numeric(df[col_ctr], errors="coerce").fillna(0.0)
else:
    df["ctr"] = np.where(
        df["impressions"] > 0, df["clicks"] / df["impressions"], 0.01
    )

df["group_id"] = (
    df[col_group].astype(str) if col_group else "group_0"
)

# Target varijabla (Definicija prilike za optimizaciju na osnovu CTR deficita)
expected_ctr_map = {
    1: 0.300,
    2: 0.150,
    3: 0.100,
    4: 0.060,
    5: 0.045,
    6: 0.035,
    7: 0.025,
    8: 0.020,
    9: 0.015,
    10: 0.010,
}
df["position_round"] = (
    df["position"].clip(lower=1, upper=20).fillna(20).astype(int)
)
df["expected_ctr"] = df["position_round"].map(
    lambda p: expected_ctr_map.get(p, 0.005)
)
df["ctr_deficit"] = (df["expected_ctr"] - df["ctr"]).clip(lower=0)

# Target (1 = Stvarna visoko-vrijedna prilika, 0 = Normalno)
vol_q75 = df["impressions"].quantile(0.75)
df["target_action"] = np.where(
    (df["position"].between(4, 15))
    & (df["impressions"] >= vol_q75)
    & (df["ctr_deficit"] >= df["expected_ctr"] * 0.30),
    1,
    0,
)

print(
    f"✓ Podaci učitani: {len(df):,} redova | Pozitivnih target klasa: {df['target_action'].sum():,}"
)

Cloning into 'FlyRank'...
remote: Enumerating objects: 328, done.
remote: Counting objects: 100% (328/328), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 328 (delta 186), reused 298 (delta 167), pack-reused 0 (from 0)
Receiving objects: 100% (328/328), 1.88 MiB | 7.22 MiB/s, done.
Resolving deltas: 100% (186/186), done.
/content/FlyRank
✓ Učitavanje skupa podataka iz: data/raw/content_refresh_anonymized.csv
✓ Podaci učitani: 30,000 redova | Pozitivnih target klasa: 1,463


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Selected model: RandomForestClassifier & HistGradientBoostingClassifier

Why this particular model?

Non-linear interactions: In SEO analytics, the relationship between position and impressions is not linear (position 4 with 50k impressions carries a completely different value than position 12 with 1k impressions). Algorithms based on decision trees naturally capture these thresholds and interactions without the need for complex transformations.

Resistance to outliers: Impressions and CPC have highly skewed distributions. Trees do not require feature normalization or scaling.

Interpretability: They allow the calculation of Permutation Importance to understand exactly which features drive the evaluation of an opportunity.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Data split design: GroupKFold / Grouped Train-Test Split based on group_id (client_id).

Why these groups?
It would be arrogant to mix pages of the same client/domain in the training and test set (Data Leakage). If the model sees one client's profile in training, it will easily guess his other pages in the test. By grouping, we ensure that the test set contains only clients that the model has never seen before.

In [5]:
# ---------------------------------------------------------
# 2. Group Split Execution & Safe Feature Selection
# ---------------------------------------------------------

# Definišemo čiste ulazne osobine bez direktnih target-leakage kolona
# Zadržavamo stabilne, osnovne metrike
base_features = ["position", "impressions", "ctr"]

# Osiguravamo da postoje u df bez korišćenja np.random
if "competition" in df.columns:
    base_features.append("competition")
if "cpc" in df.columns:
    base_features.append("cpc")

# Sanitarizacija: izbacujemo bilo kakve zabranjene reči ili target derivate
forbidden_terms = ["target", "future", "label", "next", "flag", "delta", "action_score"]
feature_cols = [
    col for col in base_features
    if col in df.columns and not any(term in col.lower() for term in forbidden_terms)
]

# Group Split po klijentima (domene)
unique_groups = df["group_id"].unique()
train_groups, test_groups = train_test_split(
    unique_groups, test_size=0.25, random_state=42
)

train_mask = df["group_id"].isin(train_groups)
test_mask = df["group_id"].isin(test_groups)

X_train, y_train = (
    df.loc[train_mask, feature_cols],
    df.loc[train_mask, "target_action"],
)
X_test, y_test = (
    df.loc[test_mask, feature_cols],
    df.loc[test_mask, "target_action"],
)

print(f"Training set: {len(X_train):,} rows | Test set (unseen groups): {len(X_test):,} rows")
print(f"Used Features: {feature_cols}")

Training set: 26,494 rows | Test set (unseen groups): 3,506 rows
Used Features: ['position', 'impressions', 'ctr', 'competition', 'cpc']


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:

test_df = df.loc[test_mask].copy()
baseline_preds = (
    (test_df["position"].between(4, 15))
    & (test_df["impressions"] >= vol_q75)
    & (test_df["ctr_deficit"] >= test_df["expected_ctr"] * 0.30)
).astype(int)

# 2. ML Model Trening (HistGradientBoosting)
model = HistGradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)

model_preds = model.predict(X_test)
model_probs = model.predict_proba(X_test)[:, 1]


# Metrike
def get_metrics(y_true, y_pred, y_prob=None):
    acc_prec = precision_score(y_true, y_pred, zero_division=0)
    acc_rec = recall_score(y_true, y_pred, zero_division=0)
    acc_f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob) if y_prob is not None else 0.5
    return {
        "Precision": round(acc_prec, 4),
        "Recall": round(acc_rec, 4),
        "F1-Score": round(acc_f1, 4),
        "ROC-AUC": round(auc, 4),
    }


baseline_metrics = get_metrics(
    y_test, baseline_preds, test_df["ctr_deficit"] / test_df["expected_ctr"]
)
ml_metrics = get_metrics(y_test, model_preds, model_probs)


comparison_df = pd.DataFrame(
    [baseline_metrics, ml_metrics],
    index=["Week-4 Baseline Rule", "Week-5 ML Model (Gradient Boosting)"],
)

print("=== MODEL VS BASELINE COMPARISON TABLE ===")
display(comparison_df)

=== MODEL VS BASELINE COMPARISON TABLE ===


,Precision,Recall,F1-Score,ROC-AUC
Week-4 Baseline Rule,1.0,1.0,1.0,0.695
Week-5 ML Model (Gradient Boosting),1.0,1.0,1.0,1.000


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:

perm_importance = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42
)
importance_df = pd.DataFrame(
    {
        "Feature": feature_cols,
        "Importance_Mean": perm_importance.importances_mean,
    }
).sort_values(by="Importance_Mean", ascending=False)

print("\n--- PERMUTATION FEATURE IMPORTANCE ---")
display(importance_df)


test_df["ml_pred"] = model_preds
test_df["error_type"] = "Correct"
test_df.loc[
    (test_df["target_action"] == 1) & (test_df["ml_pred"] == 0), "error_type"
] = "False Negative (Missed Opportunity)"
test_df.loc[
    (test_df["target_action"] == 0) & (test_df["ml_pred"] == 1), "error_type"
] = "False Positive (False Alarm)"

print("\n--- ERROR DISTRIBUTION ---")
print(test_df["error_type"].value_counts())


--- PERMUTATION FEATURE IMPORTANCE ---


,Feature,Importance_Mean
1,impressions,0.122019
0,position,0.118853
2,ctr,0.077867
3,competition,0.000000
4,cpc,0.000000



--- ERROR DISTRIBUTION ---
error_type
Correct    3506
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.